In [1]:
import os
import warnings


import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier




import mlflow
import mlflow.xgboost
import mlflow.lightgbm
from mlflow.models import infer_signature

warnings.filterwarnings("ignore")
load_dotenv()

True

In [3]:
import xgboost as xgb

In [4]:
import lightgbm as lgb

In [6]:
# Для локального стенда из docker-compose
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "admin")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "password")
raw_s3_endpoint = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")

# `minio` резолвится только внутри docker-сети. Для локального ноутбука нужен localhost.
if "minio:9000" in raw_s3_endpoint:
    raw_s3_endpoint = "http://localhost:9000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = raw_s3_endpoint

# Если на сервере включена basic-auth, задайте логин/пароль
# os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME", "admin")
# os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "password")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print("MLflow URI:", mlflow.get_tracking_uri())

MLflow URI: http://localhost:5050


In [19]:
# Быстрая проверка подключения
mlflow.search_experiments(max_results=3)

[<Experiment: artifact_location='mlflow-artifacts:/', creation_time=1779729473752, experiment_id='1', last_update_time=1779729473752, lifecycle_stage='active', name='decision_tree', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='s3://mlflow-bucket/mlflow/0', creation_time=1779728934723, experiment_id='0', last_update_time=1779728934723, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [7]:
X = pd.read_csv('../formatted_data/X_final.csv')
y = pd.read_csv('../formatted_data/y_final.csv')
RANDOM_STATE = 42
target_cols_final = y.columns.tolist()

In [ ]:
from mlflow.tracking import MlflowClient

experiment_name = "decision_tree"
artifact_location = "mlflow-artifacts:/"
client = MlflowClient()

exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
    print("Created experiment:", exp_id)
else:
    exp_id = exp.experiment_id
    print("Using existing experiment:", exp_id, "artifact_location=", exp.artifact_location)

mlflow.set_experiment(experiment_name)

registered_model_name = "decision_tree_model"

params = {
    "max_depth": 20,
    "min_samples_split": 5,
    "random_state": RANDOM_STATE
}

results = {}

Created experiment: 1


In [ ]:
for cat in target_cols_final:

    print(f"\nTARGET: {cat}")

    mask = y[cat].notna()

    X_cat = X[mask]
    y_cat = y.loc[mask, cat]

    if len(y_cat) == 0:
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X_cat,
        y_cat,
        test_size=0.2,
        random_state=RANDOM_STATE
    )

    with mlflow.start_run(
        run_name=f"dtree_{cat}"
    ):

        model = DecisionTreeClassifier(**params)

        model.fit(X_train, y_train)

        proba = model.predict_proba(X_test)[:, 1]

        roc_auc = roc_auc_score(y_test, proba)

        mlflow.log_params(params)

        mlflow.log_param("target", cat)

        mlflow.log_metric("roc_auc", float(roc_auc))

        signature = infer_signature(X_test, proba)

        registered_model_name = f"decision_tree_{cat}"

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            signature=signature,
            input_example=X_test.head(5),
            registered_model_name=registered_model_name,
        )
        new_version = model_info.registered_model_version

        client.set_registered_model_alias(
            registered_model_name,
            "prd",
            new_version
        )

        print("PRD alias ->", new_version)

        mlflow.set_tag("model_type", "DecisionTreeClassifier")
        results[cat] = {
            "roc_auc": roc_auc,
            "run_id": mlflow.active_run().info.run_id,
            "model_version": model_info.registered_model_version
        }

        print("ROC-AUC:", roc_auc)
        print("Run ID:", mlflow.active_run().info.run_id)


results_df = pd.DataFrame.from_dict(results, orient="index")

print(results_df)


========== TARGET: acute_toxicity ==========


2026/05/25 20:34:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 20:34:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/25 20:34:05 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'decision_tree_acute_toxicity' already exists. Creating a new version of this model...
2026/05/25 20:34:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: decision_tree_acute_toxicity, version 2
Created version '2' of mod

PRD alias -> 2
ROC-AUC: 0.6763550149469126
Run ID: 079a46d03e954ee8973c806c2876ce4b
🏃 View run dtree_acute_toxicity at: http://localhost:5050/#/experiments/1/runs/079a46d03e954ee8973c806c2876ce4b
🧪 View experiment at: http://localhost:5050/#/experiments/1

========== TARGET: carcinogenicity ==========


2026/05/25 20:34:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/25 20:34:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'decision_tree_carcinogenicity' already exists. Creating a new version of this model...
2026/05/25 20:34:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: decision_tree_carcinogenicity, version 2
Created version '2' of model 'decision_tree_carcinogenicity'.


PRD alias -> 2
ROC-AUC: 0.5851715686274509
Run ID: 9945d10047af4b1aa07fb12081dce45b
🏃 View run dtree_carcinogenicity at: http://localhost:5050/#/experiments/1/runs/9945d10047af4b1aa07fb12081dce45b
🧪 View experiment at: http://localhost:5050/#/experiments/1

========== TARGET: cardiotoxicity ==========


...

PRD alias -> 2
ROC-AUC: 0.6702242733851929
Run ID: cf77fad09295497ebdeb93953462f076
🏃 View run dtree_immuno_hematotoxicity at: http://localhost:5050/#/experiments/1/runs/cf77fad09295497ebdeb93953462f076
🧪 View experiment at: http://localhost:5050/#/experiments/1

========== TARGET: reprod_dev_toxicity ==========


2026/05/25 20:35:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'decision_tree_reprod_dev_toxicity' already exists. Creating a new version of this model...
2026/05/25 20:35:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: decision_tree_reprod_dev_toxicity, version 2
Created version '2' of model 'decision_tree_reprod_dev_toxicity'.


PRD alias -> 2
ROC-AUC: 0.46534653465346537
Run ID: 0990e9e8a78445b8ac617caff6432caa
🏃 View run dtree_reprod_dev_toxicity at: http://localhost:5050/#/experiments/1/runs/0990e9e8a78445b8ac617caff6432caa
🧪 View experiment at: http://localhost:5050/#/experiments/1

========== TARGET: endocrine_metabolic_tox ==========


2026/05/25 20:35:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 20:35:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/25 20:35:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'decision_tree_endocrine_metabolic_tox' already exists. Creating a new version of this model...
2026/05/25 20:35:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: decision_tree_endocrine_metabolic_tox, version 2
Created 

PRD alias -> 2
ROC-AUC: 0.6397356122663306
Run ID: 2f8ac3e61ad24c66912fc6c8493baa44
🏃 View run dtree_endocrine_metabolic_tox at: http://localhost:5050/#/experiments/1/runs/2f8ac3e61ad24c66912fc6c8493baa44
🧪 View experiment at: http://localhost:5050/#/experiments/1
                          roc_auc                            run_id  \
acute_toxicity           0.676355  079a46d03e954ee8973c806c2876ce4b   
carcinogenicity          0.585172  9945d10047af4b1aa07fb12081dce45b   
cardiotoxicity           0.677677  6727c31769c744a1a7ab103f24f1da92   
dermal_toxicity          0.649556  3f3f987ad68046d09a504a9246f9e44f   
genotoxicity             0.766710  6a5a6859d5de45048046a8c3330cb15c   
hepatotoxicity           0.653289  e12595960975413097344619b27526a4   
ocular_toxicity          0.750903  0982bf3d097343dc9d90bbcde59131fe   
oxidative_stress         0.595140  3fbdacf03d394176b34c6883f2240925   
respiratory_toxicity     0.673414  216efde5d7a144069b3db648ab214533   
neuro_sensory_toxicity   

In [12]:
runs_df = mlflow.search_runs(
    experiment_names=[experiment_name],
    order_by=["metrics.roc_auc DESC"],
)
runs_df[["run_id","metrics.roc_auc", "artifact_uri"]].head()

,run_id,metrics.roc_auc,artifact_uri
0,6a5a6859d5de45048046a8c3330cb15c,0.766710,mlflow-artifacts:/6a5a6859d5de45048046a8c3330c...
1,d218e45b31db4df3be74ce7eabe21c0a,0.766710,mlflow-artifacts:/d218e45b31db4df3be74ce7eabe2...
2,0982bf3d097343dc9d90bbcde59131fe,0.750903,mlflow-artifacts:/0982bf3d097343dc9d90bbcde591...
3,9fdf0b918f1b4a848ca61a5f2efd2ac0,0.750903,mlflow-artifacts:/9fdf0b918f1b4a848ca61a5f2efd...
4,6727c31769c744a1a7ab103f24f1da92,0.677677,mlflow-artifacts:/6727c31769c744a1a7ab103f24f1...


In [13]:
mlflow.search_runs(experiment_names=[experiment_name])

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.roc_auc,params.max_depth,params.min_samples_split,params.random_state,params.target,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.source.name,tags.model_type
0,2f8ac3e61ad24c66912fc6c8493baa44,1,FINISHED,mlflow-artifacts:/2f8ac3e61ad24c66912fc6c8493b...,2026-05-25 17:35:00.829000+00:00,2026-05-25 17:35:03.214000+00:00,0.639736,20,5,42,endocrine_metabolic_tox,kuzmndmtry,NOTEBOOK,dtree_endocrine_metabolic_tox,ML_flow.ipynb,DecisionTreeClassifier
1,0990e9e8a78445b8ac617caff6432caa,1,FINISHED,mlflow-artifacts:/0990e9e8a78445b8ac617caff643...,2026-05-25 17:34:58.639000+00:00,2026-05-25 17:35:00.788000+00:00,0.465347,20,5,42,reprod_dev_toxicity,kuzmndmtry,NOTEBOOK,dtree_reprod_dev_toxicity,ML_flow.ipynb,DecisionTreeClassifier
2,cf77fad09295497ebdeb93953462f076,1,FINISHED,mlflow-artifacts:/cf77fad09295497ebdeb93953462...,2026-05-25 17:34:56.206000+00:00,2026-05-25 17:34:58.566000+00:00,0.670224,20,5,42,immuno_hematotoxicity,kuzmndmtry,NOTEBOOK,dtree_immuno_hematotoxicity,ML_flow.ipynb,DecisionTreeClassifier
3,c6ce9d40c3ba49c2a2a6f9c8f3a00d32,1,FINISHED,mlflow-artifacts:/c6ce9d40c3ba49c2a2a6f9c8f3a0...,2026-05-25 17:34:54.097000+00:00,2026-05-25 17:34:56.178000+00:00,0.669419,20,5,42,neuro_sensory_toxicity,kuzmndmtry,NOTEBOOK,dtree_neuro_sensory_toxicity,ML_flow.ipynb,DecisionTreeClassifier
4,216efde5d7a144069b3db648ab214533,1,FINISHED,mlflow-artifacts:/216efde5d7a144069b3db648ab21...,2026-05-25 17:34:51.806000+00:00,2026-05-25 17:34:54.012000+00:00,0.673414,20,5,42,respiratory_toxicity,kuzmndmtry,NOTEBOOK,dtree_respiratory_toxicity,ML_flow.ipynb,DecisionTreeClassifier
5,3fbdacf03d394176b34c6883f2240925,1,FINISHED,mlflow-artifacts:/3fbdacf03d394176b34c6883f224...,2026-05-25 17:34:49.060000+00:00,2026-05-25 17:34:51.784000+00:00,0.595140,20,5,42,oxidative_stress,kuzmndmtry,NOTEBOOK,dtree_oxidative_stress,ML_flow.ipynb,DecisionTreeClassifier
6,0982bf3d097343dc9d90bbcde59131fe,1,FINISHED,mlflow-artifacts:/0982bf3d097343dc9d90bbcde591...,2026-05-25 17:34:45.361000+00:00,2026-05-25 17:34:48.973000+00:00,0.750903,20,5,42,ocular_toxicity,kuzmndmtry,NOTEBOOK,dtree_ocular_toxicity,ML_flow.ipynb,DecisionTreeClassifier
7,e12595960975413097344619b27526a4,1,FINISHED,mlflow-artifacts:/e12595960975413097344619b275...,2026-05-25 17:34:39.858000+00:00,2026-05-25 17:34:45.224000+00:00,0.653289,20,5,42,hepatotoxicity,kuzmndmtry,NOTEBOOK,dtree_hepatotoxicity,ML_flow.ipynb,DecisionTreeClassifier
8,6a5a6859d5de45048046a8c3330cb15c,1,FINISHED,mlflow-artifacts:/6a5a6859d5de45048046a8c3330c...,2026-05-25 17:34:37.006000+00:00,2026-05-25 17:34:39.759000+00:00,0.766710,20,5,42,genotoxicity,kuzmndmtry,NOTEBOOK,dtree_genotoxicity,ML_flow.ipynb,DecisionTreeClassifier
9,3f3f987ad68046d09a504a9246f9e44f,1,FINISHED,mlflow-artifacts:/3f3f987ad68046d09a504a9246f9...,2026-05-25 17:34:34.245000+00:00,2026-05-25 17:34:36.869000+00:00,0.649556,20,5,42,dermal_toxicity,kuzmndmtry,NOTEBOOK,dtree_dermal_toxicity,ML_flow.ipynb,DecisionTreeClassifier


In [14]:
loaded_model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@prd")

sample_pred = loaded_model.predict(X_test.head(3))
sample_pred


array([0., 0., 0.])

Random Forest

In [21]:
from mlflow.tracking import MlflowClient

experiment_name = "random_forest"
artifact_location = "mlflow-artifacts:/"
client = MlflowClient()

exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
    print("Created experiment:", exp_id)
else:
    exp_id = exp.experiment_id
    print("Using existing experiment:", exp_id, "artifact_location=", exp.artifact_location)

mlflow.set_experiment(experiment_name)

registered_model_name = "random_forest_model"

rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "criterion": ["gini"],
    "max_features": ["sqrt"]
}

results = []

Created experiment: 2


In [23]:
for cat in target_cols_final:
    print(f"\nTARGET: {cat}")

    mask = y[cat].notna()

    X_cat = X[mask]
    y_cat = y.loc[mask, cat]

    if len(y_cat) == 0:
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X_cat,
        y_cat,
        test_size=0.2,
        random_state=RANDOM_STATE
    )

    with mlflow.start_run(run_name=f"rf_{cat}"):
        rf = RandomForestClassifier(
            random_state=42,
            n_jobs=-1
            )

        grid_search = GridSearchCV(

            estimator=rf,

            param_grid=rf_param_grid,

            cv=3,

            scoring="roc_auc",

            n_jobs=-1

        )

        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_

        proba = best_model.predict_proba(X_test)[:, 1]

        roc_auc = roc_auc_score(y_test, proba)


        best_params = grid_search.best_params_

        metrics = {

            "roc_auc_test": float(roc_auc),

            "best_cv_score": float(grid_search.best_score_)

        }

        mlflow.log_params(best_params)

        mlflow.log_param("target", cat)

        mlflow.log_metrics(metrics)

        mlflow.set_tags({

            "model_type": "RandomForestClassifier",

            "target": cat

        })


        signature = infer_signature(X_test, proba)

        registered_model_name = f"random_forest_{cat}"

        model_info = mlflow.sklearn.log_model(

            sk_model=best_model,

            name="model",

            signature=signature,

            input_example=X_test.head(5),

            registered_model_name=registered_model_name,

        )

        new_version = model_info.registered_model_version

        client.set_registered_model_alias(

            registered_model_name,

            "prd",

            new_version

        )

        # feature importance artifact

        importance_df = pd.DataFrame({

            "feature": X_train.columns,

            "importance": best_model.feature_importances_

        }).sort_values("importance", ascending=False)

        importance_path = f"feature_importance_{cat}.csv"

        importance_df.to_csv(

            importance_path,

            index=False

        )

        mlflow.log_artifact(importance_path)



        results.append({

            "category": cat,

            "roc_auc_test": roc_auc,

            "best_cv_score": grid_search.best_score_,

            "best_params": best_params,

            "run_id": mlflow.active_run().info.run_id,

            "model_version": new_version

        })

        print("ROC-AUC TEST:", roc_auc)

        print("Best CV Score:", grid_search.best_score_)

        print("Best Params:", best_params)



results_rf = pd.DataFrame(results)

print(results_rf)


TARGET: acute_toxicity


2026/05/27 14:40:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/27 14:40:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'random_forest_acute_toxicity'.
2026/05/27 14:40:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: random_forest_acute_toxicity, version 1
Created version '1' of model 'random_forest_acute_toxicity'.


ROC-AUC TEST: 0.8409648489846407
Best CV Score: 0.8230414435494757
Best Params: {'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
🏃 View run rf_acute_toxicity at: http://localhost:5050/#/experiments/2/runs/e2823d1363df44cf880f0aba558ae31e
🧪 View experiment at: http://localhost:5050/#/experiments/2

TARGET: carcinogenicity


2026/05/27 14:40:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/27 14:40:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'random_forest_carcinogenicity'.
2026/05/27 14:40:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: random_forest_carcinogenicity, version 1
Created version '1' of model 'random_forest_carcinogenicity'.


ROC-AUC TEST: 0.7314600840336134
Best CV Score: 0.7258832284353023
Best Params: {'criterion': 'gini', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
🏃 View run rf_carcinogenicity at: http://localhost:5050/#/experiments/2/runs/6d376d64bce044a2b3bedf2e6a3d2bce
🧪 View experiment at: http://localhost:5050/#/experiments/2

TARGET: cardiotoxicity


...

ROC-AUC TEST: 0.8145845427454623
Best CV Score: 0.7890370803782852
Best Params: {'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
🏃 View run rf_immuno_hematotoxicity at: http://localhost:5050/#/experiments/2/runs/04061456509a4d5a8eb59dca70135011
🧪 View experiment at: http://localhost:5050/#/experiments/2

TARGET: reprod_dev_toxicity


2026/05/27 14:58:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/27 14:58:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'random_forest_reprod_dev_toxicity'.
2026/05/27 14:58:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: random_forest_reprod_dev_toxicity, version 1
Created version '1' of model 'random_forest_reprod_dev_toxicity'.


ROC-AUC TEST: 0.8816831683168317
Best CV Score: 0.7740697619374091
Best Params: {'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
🏃 View run rf_reprod_dev_toxicity at: http://localhost:5050/#/experiments/2/runs/563be08be3604bee88eacdc88cfec3ca
🧪 View experiment at: http://localhost:5050/#/experiments/2

TARGET: endocrine_metabolic_tox


2026/05/27 14:59:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/27 14:59:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'random_forest_endocrine_metabolic_tox'.
2026/05/27 14:59:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: random_forest_endocrine_metabolic_tox, version 1
Created version '1' of model 'random_forest_endocrine_metabolic_tox'.


ROC-AUC TEST: 0.7725774147087947
Best CV Score: 0.7585320103871132
Best Params: {'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
🏃 View run rf_endocrine_metabolic_tox at: http://localhost:5050/#/experiments/2/runs/95653f266e3c48e48ffde3e3c5fce46b
🧪 View experiment at: http://localhost:5050/#/experiments/2
                   category  roc_auc_test  best_cv_score  \
0            acute_toxicity      0.840965       0.823041   
1           carcinogenicity      0.731460       0.725883   
2            cardiotoxicity      0.903505       0.895438   
3           dermal_toxicity      0.801262       0.813160   
4              genotoxicity      0.911589       0.903342   
5            hepatotoxicity      0.836027       0.803080   
6           ocular_toxicity      0.896080       0.909220   
7          oxidative_stress      0.805326       0.801676   
8      respiratory_toxicity      0.882922       0.829528   
9    neuro_

Бустинги

In [8]:
from mlflow.tracking import MlflowClient

experiment_name = "lightGBM"
artifact_location = "mlflow-artifacts:/"
client = MlflowClient()

exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
    print("Created experiment:", exp_id)
else:
    exp_id = exp.experiment_id
    print("Using existing experiment:", exp_id, "artifact_location=", exp.artifact_location)

mlflow.set_experiment(experiment_name)

registered_model_name = "lightGBM_model"

lgb_param_grid = {
    "num_leaves": [31, 63],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
    "subsample": [0.8, 1.0]
}

results = []

Using existing experiment: 3 artifact_location= mlflow-artifacts:/


In [10]:
for cat in y.columns.tolist():

    print(f"\nTARGET: {cat}")

    mask = y[cat].notna()

    X_cat = X[mask]
    y_cat = y.loc[mask, cat]

    if len(y_cat) == 0:
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X_cat,
        y_cat,
        test_size=0.2,
        random_state=42,
        stratify=y_cat
    )

    with mlflow.start_run(run_name=f"lgbm_{cat}"):

        lgb_model = lgb.LGBMClassifier(
            random_state=42,
            n_jobs=-1,
            device="cpu"
        )

        lgb_grid = GridSearchCV(
            estimator=lgb_model,
            param_grid=lgb_param_grid,
            cv=3,
            scoring="roc_auc",
            n_jobs=-1,
            verbose=2
        )

        lgb_grid.fit(X_train, y_train)

        lgb_best = lgb_grid.best_estimator_

        proba = lgb_best.predict_proba(X_test)[:, 1]

        roc_auc = roc_auc_score(y_test, proba)

        best_params = lgb_grid.best_params_

        metrics = {
            "roc_auc_test": float(roc_auc),
            "best_cv_score": float(lgb_grid.best_score_)
        }

        mlflow.log_params(best_params)

        mlflow.log_param("target", cat)

        mlflow.log_metrics(metrics)

        mlflow.set_tags({
            "model_type": "LightGBM",
            "target": cat
        })

        signature = infer_signature(X_test, proba)

        registered_model_name = f"lightgbm_{cat}"

        model_info = mlflow.lightgbm.log_model(
            lgb_model=lgb_best,
            artifact_path="model",
            signature=signature,
            input_example=X_test.head(5),
            registered_model_name=registered_model_name,
        )

        new_version = model_info.registered_model_version

        client.set_registered_model_alias(
            registered_model_name,
            "prd",
            new_version
        )

        importance_df = pd.DataFrame({
            "feature": X_train.columns,
            "importance": lgb_best.feature_importances_
        }).sort_values("importance", ascending=False)

        importance_path = f"lgbm_feature_importance_{cat}.csv"

        importance_df.to_csv(
            importance_path,
            index=False
        )

        mlflow.log_artifact(importance_path)

        results.append({
            "category": cat,
            "roc_auc_test": roc_auc,
            "best_cv_score": lgb_grid.best_score_,
            "best_params": best_params,
            "run_id": mlflow.active_run().info.run_id,
            "model_version": new_version
        })

        print("ROC-AUC TEST:", roc_auc)
        print("Best CV Score:", lgb_grid.best_score_)
        print("Best Params:", best_params)

results_lgbm = pd.DataFrame(results)

print(results_lgbm)


TARGET: acute_toxicity
Fitting 3 folds for each of 16 candidates, totalling 48 fits
[LightGBM] [Info] Number of positive: 1239, number of negative: 2853
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012136 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12618
[LightGBM] [Info] Number of data points in the train set: 4092, number of used features: 78
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.302786 -> initscore=-0.834066
[LightGBM] [Info] Start training from score -0.834066
[LightGBM] [Info] Number of positive: 1239, number of negative: 2853
[LightGBM] [Info] Number of positive: 1238, number of negative: 2854
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020218 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12613
[LightGBM] [Info] Number of data points in the train set: 4092, number of used features:

2026/05/27 16:08:26 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'lightgbm_acute_toxicity'.
2026/05/27 16:08:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: lightgbm_acute_toxicity, version 1
Created version '1' of model 'lightgbm_acute_toxicity'.


ROC-AUC TEST: 0.8608340870264294
Best CV Score: 0.8414241127645447
Best Params: {'learning_rate': 0.05, 'n_estimators': 200, 'num_leaves': 31, 'subsample': 0.8}
🏃 View run lgbm_acute_toxicity at: http://localhost:5050/#/experiments/3/runs/d4300da6389540678b9dbb7931e170ba
🧪 View experiment at: http://localhost:5050/#/experiments/3

TARGET: carcinogenicity
Fitting 3 folds for each of 16 candidates, totalling 48 fits
[LightGBM] [Info] Number of positive: 552, number of negative: 363
[LightGBM] [Info] Number of positive: 553, number of negative: 362
[LightGBM] [Info] Number of positive: 553, number of negative: 363
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005372 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8692
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007571 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total

2026/05/27 16:09:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'lightgbm_carcinogenicity'.
2026/05/27 16:09:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: lightgbm_carcinogenicity, version 1
Created version '1' of model 'lightgbm_carcinogenicity'.


ROC-AUC TEST: 0.7495050904977376
Best CV Score: 0.7104196425137482
Best Params: {'learning_rate': 0.05, 'n_estimators': 200, 'num_leaves': 31, 'subsample': 0.8}
🏃 View run lgbm_carcinogenicity at: http://localhost:5050/#/experiments/3/runs/18f01acfa2f6482d8909f697e54ffd4d
🧪 View experiment at: http://localhost:5050/#/experiments/3

TARGET: cardiotoxicity
Fitting 3 folds for each of 16 candidates, totalling 48 fits
[LightGBM] [Info] Number of positive: 11811, number of negative: 159381
[LightGBM] [Info] Number of positive: 11812, number of negative: 159380
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.039557 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Number of positive: 11811, number of negative: 159381
[LightGBM] [Info] Total Bins 14574
[LightGBM] [Info] Number of data points in the train set: 171192, number of used features: 78
[LightGBM] 

...

ROC-AUC TEST: 0.8056807120011125
Best CV Score: 0.7983594732503502
Best Params: {'learning_rate': 0.05, 'n_estimators': 200, 'num_leaves': 31, 'subsample': 0.8}
🏃 View run lgbm_immuno_hematotoxicity at: http://localhost:5050/#/experiments/3/runs/837071b44c344b318637914ee93bfcc4
🧪 View experiment at: http://localhost:5050/#/experiments/3

TARGET: reprod_dev_toxicity
Fitting 3 folds for each of 16 candidates, totalling 48 fits
[LightGBM] [Info] Number of positive: 543, number of negative: 22
[LightGBM] [Info] Number of positive: 543, number of negative: 22
[LightGBM] [Info] Number of positive: 544, number of negative: 22
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013918 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6020
[LightGBM] [Info] Number of data points in the train set: 565, number of used features: 77
[LightGBM] [Info] [bin

2026/05/27 16:24:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'lightgbm_reprod_dev_toxicity'.
2026/05/27 16:24:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: lightgbm_reprod_dev_toxicity, version 1
Created version '1' of model 'lightgbm_reprod_dev_toxicity'.


ROC-AUC TEST: 0.8290441176470588
Best CV Score: 0.7490844794811585
Best Params: {'learning_rate': 0.05, 'n_estimators': 200, 'num_leaves': 63, 'subsample': 0.8}
🏃 View run lgbm_reprod_dev_toxicity at: http://localhost:5050/#/experiments/3/runs/72be6a7ea7ca4c91981dcace190c20b2
🧪 View experiment at: http://localhost:5050/#/experiments/3

TARGET: endocrine_metabolic_tox
Fitting 3 folds for each of 16 candidates, totalling 48 fits
[LightGBM] [Info] Number of positive: 1305, number of negative: 2927
[LightGBM] [Info] Number of positive: 1305, number of negative: 2927
[LightGBM] [Info] Number of positive: 1306, number of negative: 2926
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020833 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12469
[LightGBM] [Info] Number of data points in the train set: 4232, number of used features: 78
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.308365 -> initscore=-0.

2026/05/27 16:26:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'lightgbm_endocrine_metabolic_tox'.
2026/05/27 16:26:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: lightgbm_endocrine_metabolic_tox, version 1
Created version '1' of model 'lightgbm_endocrine_metabolic_tox'.


ROC-AUC TEST: 0.7620700239986605
Best CV Score: 0.7504165074793366
Best Params: {'learning_rate': 0.05, 'n_estimators': 200, 'num_leaves': 31, 'subsample': 0.8}
🏃 View run lgbm_endocrine_metabolic_tox at: http://localhost:5050/#/experiments/3/runs/06671b9fb0b6476d952d62e4c1401faa
🧪 View experiment at: http://localhost:5050/#/experiments/3
                   category  roc_auc_test  best_cv_score  \
0            acute_toxicity      0.860834       0.841424   
1           carcinogenicity      0.749505       0.710420   
2            cardiotoxicity      0.915984       0.906666   
3           dermal_toxicity      0.801439       0.794047   
4              genotoxicity      0.912991       0.905435   
5            hepatotoxicity      0.788163       0.809871   
6           ocular_toxicity      0.903893       0.918829   
7          oxidative_stress      0.798321       0.791754   
8      respiratory_toxicity      0.887742       0.846762   
9    neuro_sensory_toxicity      0.837156       0.850895   

In [11]:
from mlflow.tracking import MlflowClient

experiment_name = "XGB"
artifact_location = "mlflow-artifacts:/"
client = MlflowClient()

exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
    print("Created experiment:", exp_id)
else:
    exp_id = exp.experiment_id
    print("Using existing experiment:", exp_id, "artifact_location=", exp.artifact_location)

mlflow.set_experiment(experiment_name)

registered_model_name = "XGB_model"

xgb_param_grid = {
    "max_depth": [6, 10],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

results = []

Created experiment: 4


In [12]:
for cat in y.columns.tolist():

    print(f"\nTARGET: {cat}")

    mask = y[cat].notna()

    X_cat = X[mask]
    y_cat = y.loc[mask, cat]

    if len(y_cat) == 0:
        continue

    if y_cat.nunique() < 2:
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X_cat,
        y_cat,
        test_size=0.2,
        random_state=42,
        stratify=y_cat
    )

    with mlflow.start_run(run_name=f"xgb_{cat}"):

        xgb_model = xgb.XGBClassifier(
            random_state=42,
            n_jobs=-1,
            eval_metric="logloss",
            tree_method="hist"
        )

        xgb_grid = GridSearchCV(
            estimator=xgb_model,
            param_grid=xgb_param_grid,
            cv=3,
            scoring="roc_auc",
            n_jobs=-1,
            verbose=2
        )

        xgb_grid.fit(X_train, y_train)

        xgb_best = xgb_grid.best_estimator_

        proba = xgb_best.predict_proba(X_test)[:, 1]

        roc_auc = roc_auc_score(y_test, proba)

        best_params = xgb_grid.best_params_

        metrics = {
            "roc_auc_test": float(roc_auc),
            "best_cv_score": float(xgb_grid.best_score_)
        }

        mlflow.log_params(best_params)

        mlflow.log_param("target", cat)

        mlflow.log_metrics(metrics)

        mlflow.set_tags({
            "model_type": "XGBoost",
            "target": cat
        })

        signature = infer_signature(X_test, proba)

        registered_model_name = f"xgboost_{cat}"

        model_info = mlflow.xgboost.log_model(
            xgb_model=xgb_best,
            artifact_path="model",
            signature=signature,
            input_example=X_test.head(5),
            registered_model_name=registered_model_name,
        )

        new_version = model_info.registered_model_version

        client.set_registered_model_alias(
            registered_model_name,
            "prd",
            new_version
        )


        importance_df = pd.DataFrame({
            "feature": X_train.columns,
            "importance": xgb_best.feature_importances_
        }).sort_values("importance", ascending=False)

        importance_path = f"xgb_feature_importance_{cat}.csv"

        importance_df.to_csv(
            importance_path,
            index=False
        )

        mlflow.log_artifact(importance_path)

        results.append({
            "category": cat,
            "roc_auc_test": roc_auc,
            "best_cv_score": xgb_grid.best_score_,
            "best_params": best_params,
            "run_id": mlflow.active_run().info.run_id,
            "model_version": new_version
        })

        print("ROC-AUC TEST:", roc_auc)
        print("Best CV Score:", xgb_grid.best_score_)
        print("Best Params:", best_params)

results_xgb = pd.DataFrame(results)

print(results_xgb)


TARGET: acute_toxicity
Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.6s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   1.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   1.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total time=   2.7s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=400, subsample=0.8; total 

2026/05/27 16:30:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'xgboost_acute_toxicity'.
2026/05/27 16:30:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgboost_acute_toxicity, version 1
Created version '1' of model 'xgboost_acute_toxicity'.


ROC-AUC TEST: 0.8601386795296955
Best CV Score: 0.8442705852027673
Best Params: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}
🏃 View run xgb_acute_toxicity at: http://localhost:5050/#/experiments/4/runs/587a6036454d4c55a4ed9ae7cc28b052
🧪 View experiment at: http://localhost:5050/#/experiments/4

TARGET: carcinogenicity
Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   0.8s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   0.9s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   0.9s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   0.9s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8;

2026/05/27 16:30:43 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'xgboost_carcinogenicity'.
2026/05/27 16:30:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgboost_carcinogenicity, version 1
Created version '1' of model 'xgboost_carcinogenicity'.


ROC-AUC TEST: 0.7465002828054298
Best CV Score: 0.7112624412772007
Best Params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 400, 'subsample': 0.8}
🏃 View run xgb_carcinogenicity at: http://localhost:5050/#/experiments/4/runs/fa86a3f299494593a4fb25aba83beb1a
🧪 View experiment at: http://localhost:5050/#/experiments/4

TARGET: cardiotoxicity
Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=  12.9s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=  13.2s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=  13.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=  13.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8;

...

ROC-AUC TEST: 0.8059032123487693
Best CV Score: 0.8029370304197198
Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}
🏃 View run xgb_immuno_hematotoxicity at: http://localhost:5050/#/experiments/4/runs/7a5de01358264fe9987434eb9322fc1f
🧪 View experiment at: http://localhost:5050/#/experiments/4

TARGET: reprod_dev_toxicity
Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   0.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   0.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   0.3s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   0.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, sub

2026/05/27 16:49:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'xgboost_reprod_dev_toxicity'.
2026/05/27 16:49:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgboost_reprod_dev_toxicity, version 1
Created version '1' of model 'xgboost_reprod_dev_toxicity'.


ROC-AUC TEST: 0.826593137254902
Best CV Score: 0.7959215554722392
Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 200, 'subsample': 0.8}
🏃 View run xgb_reprod_dev_toxicity at: http://localhost:5050/#/experiments/4/runs/a6aa94898dd2435889d30500c2b8b689
🧪 View experiment at: http://localhost:5050/#/experiments/4

TARGET: endocrine_metabolic_tox
Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   2.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   2.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=0.8; total time=   2.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, subsample=1.0; total time=   2.0s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=6, n_estimators=200, s

2026/05/27 16:50:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'xgboost_endocrine_metabolic_tox'.
2026/05/27 16:50:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgboost_endocrine_metabolic_tox, version 1
Created version '1' of model 'xgboost_endocrine_metabolic_tox'.


ROC-AUC TEST: 0.7632290290774468
Best CV Score: 0.7559168255437104
Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}
🏃 View run xgb_endocrine_metabolic_tox at: http://localhost:5050/#/experiments/4/runs/5f33333498ee4ba29fb5d576522063d1
🧪 View experiment at: http://localhost:5050/#/experiments/4
                   category  roc_auc_test  best_cv_score  \
0            acute_toxicity      0.860139       0.844271   
1           carcinogenicity      0.746500       0.711262   
2            cardiotoxicity      0.922909       0.913771   
3           dermal_toxicity      0.800024       0.797175   
4              genotoxicity      0.916615       0.909432   
5            hepatotoxicity      0.800923       0.812566   
6           ocular_toxicity      0.908804       0.921868   
7          oxidative_stress      0.804344       0.804632   
8      respiratory_toxicity      0.875079       0.854336   
9    neuro_sensory_toxicity      0.85